## 1. Setup

Install dependencies and configure the environment.

## 1. Setup

Install dependencies and configure the environment.

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q python-dotenv google-genai google-adk matplotlib

print("✅ Dependencies installed")

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv(project_root / ".env")

print(f"✅ Project root: {project_root}")

In [ ]:
# Configure Google Gemini API
# Get your free API key at: https://aistudio.google.com/app/apikey

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

if GOOGLE_API_KEY:
    print("✅ Google Gemini API configured")
    print(f"   Key: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print("")
    print("Please set your API key:")
    print("  1. Get free key: https://aistudio.google.com/app/apikey")
    print("  2. Create a .env file with: GOOGLE_API_KEY=your-key-here")

---

## 2. Understanding ODD & COD

### 🎯 ODD - Operational Design Domain

The **ODD** defines the *safe operating envelope* for a robot.

| Category | Examples |
|----------|----------|
| **Environment** | Lighting, terrain, indoor/outdoor |
| **Dynamic** | Max speed (2.5 m/s), acceleration, pitch/roll |
| **Safety** | Obstacle density, proximity requirements |

### 📊 COD - Current Operating Domain

The **COD** captures *what actually happened* during robot operation.

- **Per-window measurements**: Observed conditions in each time window
- **Compliance check**: Is COD ⊆ ODD? (COD contained within ODD)

---

## 3. The 6-Agent Pipeline

```
┌─────────────────┐
│  OddSpecAgent   │  Parse natural language ODD → structured spec
└────────┬────────┘
         │
         ▼
┌─────────────────────────────────────────┐
│  Parallel Sensor Analysis               │
│  ┌───────────┐ ┌──────────┐ ┌─────────┐ │
│  │Perception │ │  Motion  │ │Collision│ │
│  │  Agent    │ │  Agent   │ │  Agent  │ │
│  └───────────┘ └──────────┘ └─────────┘ │
└────────┬───────────┬───────────┬────────┘
         │           │           │
         ▼           ▼           ▼
┌─────────────────────────────────────────┐
│           EvaluatorAgent                │
│  Constructs COD, computes compliance    │
└────────────────┬────────────────────────┘
                 │
                 ▼
┌─────────────────────────────────────────┐
│            ReportAgent                  │
│  Executive summary & recommendations    │
└─────────────────────────────────────────┘
```

---

## 4. Available Scenarios

Let's see what test data is available.

In [ ]:
# List available scenarios
data_dir = project_root / "data"

print("📁 Production Scenarios:")
prod_chunks = data_dir / "production" / "chunks"
if prod_chunks.exists():
    for scenario in sorted(prod_chunks.iterdir()):
        if scenario.is_dir():
            windows = list(scenario.glob("window_*"))
            print(f"   {scenario.name}: {len(windows)} windows")

print("\n📁 Test Scenarios (2-window quick tests):")
test_dir = data_dir / "test"
if test_dir.exists():
    for scenario in sorted(test_dir.iterdir()):
        if scenario.is_dir() and not scenario.name.startswith('.'):
            windows = list(scenario.glob("window_*"))
            print(f"   {scenario.name}: {len(windows)} windows")

In [ ]:
# Select a scenario
SCENARIO = "sim_2win"  # Options: sim_2win, real_2win, real_173442_2win, etc.

# Determine scenario path
if SCENARIO.endswith("_2win"):
    scenario_path = data_dir / "test" / SCENARIO
else:
    scenario_path = data_dir / "production" / "chunks" / SCENARIO

if scenario_path.exists():
    windows = sorted(scenario_path.glob("window_*"))
    print(f"✅ Selected: {SCENARIO}")
    print(f"   Path: {scenario_path}")
    print(f"   Windows: {len(windows)}")
else:
    print(f"❌ Scenario not found: {scenario_path}")

---

## 5. Explore Window Data

Each window contains camera images, LiDAR BEV, and motion JSON.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Examine first window
window_dir = windows[0]
print(f"📁 Window: {window_dir.name}")
print("\nContents:")
for f in sorted(window_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name}: {size_kb:.1f} KB")

In [ ]:
# Display camera images
camera_images = sorted(window_dir.glob("camera_*.jpg"))

if camera_images:
    fig, axes = plt.subplots(1, min(3, len(camera_images)), figsize=(15, 5))
    if len(camera_images) == 1:
        axes = [axes]
    
    for ax, img_path in zip(axes, camera_images[:3]):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(img_path.name)
        ax.axis('off')
    
    plt.suptitle(f"Camera Images - {window_dir.name}")
    plt.tight_layout()
    plt.show()
else:
    print("No camera images found")

In [ ]:
# Display BEV (Bird's Eye View) channels
bev_channels = ['occupancy', 'height', 'roughness']
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, channel in zip(axes, bev_channels):
    bev_path = window_dir / f"bev_{channel}.png"
    if bev_path.exists():
        img = Image.open(bev_path)
        ax.imshow(img, cmap='gray' if channel != 'occupancy' else None)
        ax.set_title(f"BEV {channel.title()}")
    else:
        ax.text(0.5, 0.5, f"No {channel}", ha='center')
        ax.set_title(f"BEV {channel.title()} (missing)")
    ax.axis('off')

plt.suptitle(f"LiDAR BEV Channels - {window_dir.name}")
plt.tight_layout()
plt.show()

print("\n📝 BEV Channel Semantics:")
print("   Occupancy: Obstacles only (ground filtered out)")
print("   Height: ALL points including ground (terrain elevation)")
print("   Roughness: ALL points (terrain height variance)")

In [ ]:
# Load and display motion data
motion_file = window_dir / "motion.json"

if motion_file.exists():
    with open(motion_file) as f:
        motion_data = json.load(f)
    
    print("📊 Motion Data Summary:")
    print(f"   Frames: {len(motion_data)}")
    
    # Extract key metrics
    speeds = [m.get('derived_speed', 0) for m in motion_data if m.get('derived_speed') is not None]
    
    if speeds:
        print(f"   Speed range: {min(speeds):.3f} - {max(speeds):.3f} m/s")
        print(f"   Avg speed: {sum(speeds)/len(speeds):.3f} m/s")
    
    # Show first frame structure
    print("\n   First frame keys:")
    for key in sorted(motion_data[0].keys()):
        print(f"     - {key}")
else:
    print("No motion.json found")

---

## 6. Run Analysis Pipeline

Use the command-line runner for full analysis.

In [ ]:
# Show available options
print("🔧 Analysis Runner Options:")
print("")
print("Interactive mode (recommended for first run):")
print("  python scripts/run_odd_analysis.py")
print("")
print("Direct scenario run:")
print("  python scripts/run_odd_analysis.py --scenario sim_2win")
print("  python scripts/run_odd_analysis.py --scenario real_2win")
print("")
print("Custom ODD:")
print('  python scripts/run_odd_analysis.py --scenario sim_2win --odd "Indoor robot, max 1.5 m/s"')
print("")
print("Disable knowledge seeding:")
print("  python scripts/run_odd_analysis.py --scenario sim_2win --no-knowledge")

In [ ]:
# Run analysis on selected scenario
# This will take ~2 minutes for 2 windows

import subprocess

print(f"🚀 Running analysis on: {SCENARIO}")
print("   This may take 2-3 minutes...")
print("")

result = subprocess.run(
    [sys.executable, "scripts/run_odd_analysis.py", "--scenario", SCENARIO],
    cwd=project_root,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.returncode != 0:
    print("\n❌ Error:")
    print(result.stderr)

---

## 7. Review Results

Find and examine the analysis results.

In [ ]:
# Find latest results
report_dir = data_dir / "report_candidates"

if report_dir.exists():
    timestamps = sorted(report_dir.iterdir(), reverse=True)
    if timestamps:
        latest = timestamps[0]
        print(f"📁 Latest results: {latest.name}")
        
        for scenario_dir in latest.iterdir():
            result_file = scenario_dir / "full_result.json"
            if result_file.exists():
                print(f"   ✅ {scenario_dir.name}/full_result.json")
else:
    print("No results found yet. Run the analysis first.")

In [ ]:
# Load and display results summary
def load_latest_result(scenario_name):
    """Load the most recent result for a scenario."""
    report_dir = project_root / "data" / "report_candidates"
    
    for timestamp_dir in sorted(report_dir.iterdir(), reverse=True):
        result_path = timestamp_dir / scenario_name / "full_result.json"
        if result_path.exists():
            with open(result_path) as f:
                return json.load(f), result_path
    return None, None

result, result_path = load_latest_result(SCENARIO)

if result:
    print(f"📊 Analysis Results: {SCENARIO}")
    print(f"   Source: {result_path}")
    print("")
    
    # Overall verdict
    if 'evaluator' in result and 'output' in result['evaluator']:
        eval_output = result['evaluator']['output']
        verdict = eval_output.get('verdict', 'Unknown')
        confidence = eval_output.get('confidence', 0)
        print(f"   🎯 Verdict: {verdict}")
        print(f"   📈 Confidence: {confidence}%")
    
    # Cost
    if 'cost' in result:
        print(f"   💰 Total Cost: ${result['cost'].get('total_cost', 0):.4f}")
else:
    print(f"No results found for {SCENARIO}")

---

## 8. Generate HTML Report

Create a visual report from the analysis results.

In [ ]:
# Generate HTML report
if result_path:
    output_path = project_root / "docs" / "reports" / f"{SCENARIO}_report.html"
    
    print(f"📄 Generating HTML report...")
    print(f"   Input: {result_path}")
    print(f"   Output: {output_path}")
    print("")
    
    gen_result = subprocess.run(
        [
            sys.executable, "scripts/generate_html_report.py",
            "--input", str(result_path),
            "--scenario-dir", str(scenario_path),
            "--output", str(output_path)
        ],
        cwd=project_root,
        capture_output=True,
        text=True
    )
    
    if gen_result.returncode == 0:
        print("✅ Report generated!")
        print(f"   View at: file://{output_path}")
    else:
        print("❌ Error generating report:")
        print(gen_result.stderr)
else:
    print("No results to generate report from. Run analysis first.")

---

## 9. Cost Estimation

Estimate API costs before running large batches.

In [ ]:
# Cost estimation
COST_PER_WINDOW = 0.025  # Approximate cost in USD

print("💰 Cost Estimation")
print("")
print(f"Approximate cost per window: ${COST_PER_WINDOW}")
print("")

# Count windows in production scenarios
if prod_chunks.exists():
    for scenario in sorted(prod_chunks.iterdir()):
        if scenario.is_dir():
            n_windows = len(list(scenario.glob("window_*")))
            cost = n_windows * COST_PER_WINDOW
            print(f"   {scenario.name}: {n_windows} windows ≈ ${cost:.2f}")

---

## 10. Next Steps

### Explore Further

1. **Run different scenarios**: Try `real_2win` or `real_173442_2win`
2. **Custom ODD**: Define your own operational constraints
3. **Production runs**: Analyze full 16-window scenarios

### Documentation

- [Architecture Overview](../docs/index.html)
- [Data Generation Guide](../docs/DATA_GENERATION.md)
- [Scenario Descriptions](../docs/SCENARIO_DESCRIPTIONS.md)

### Agent Knowledge

The agents use knowledge documents in `docs/agent_knowledge/`:
- `odd_cod_fundamentals.md` - Core ODD/COD concepts
- `sensor_interpretation.md` - BEV/camera/IMU patterns
- `collision_detection.md` - Collision detection guidance

In [ ]:
# Install dependencies (run once)
import sys
!{sys.executable} -m pip install -q python-dotenv google-genai google-adk matplotlib

print("✅ Dependencies installed")

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Load environment variables
load_dotenv(project_root / ".env")

print(f"✅ Project root: {project_root}")

In [ ]:
# Configure Google Gemini API
# Get your free API key at: https://aistudio.google.com/app/apikey

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

if GOOGLE_API_KEY:
    print("✅ Google Gemini API configured")
    print(f"   Key: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print("")
    print("Please set your API key:")
    print("  1. Get free key: https://aistudio.google.com/app/apikey")
    print("  2. Create a .env file with: GOOGLE_API_KEY=your-key-here")
    print("     OR uncomment and set below:")
    print("")
    # Uncomment to set directly (not recommended for sharing):
    # os.environ['GOOGLE_API_KEY'] = 'your-key-here'
    # GOOGLE_API_KEY = os.environ['GOOGLE_API_KEY']

---

## 2. Understanding ODD & COD

Before running the analysis, let's understand the key concepts.

### 🎯 ODD - Operational Design Domain

The **ODD** defines the *safe operating envelope* for a robot—the conditions under which it's designed to function correctly.

| Category | Examples |
|----------|----------|
| **Environment** | Lighting (bright/dim), terrain (smooth/rough), indoor/outdoor |
| **Dynamic** | Max speed (2.5 m/s), acceleration (10 m/s²), pitch/roll limits |
| **Safety** | Obstacle density, human/animal proximity |

### 📊 COD - Current Operating Domain

The **COD** captures *what actually happened* during robot operation—the observed conditions across all ODD axes.

- **Per-window measurements**: What was observed in each time window
- **Region construction**: Aggregated operational envelope
- **Compliance check**: Is COD ⊆ ODD? (COD contained within ODD)

### Verdict Types

| Verdict | Meaning |
|---------|--------|
| ✅ **IN_ODD** | COD fully within ODD - safe operation |
| ⚠️ **BOUNDARY** | COD at or near edge of ODD - caution |
| ❌ **OUT_ODD** | COD exceeds ODD limits - violation |

---

## 3. The 6-Agent Pipeline

Our analysis uses 6 specialized AI agents built with Google ADK + Gemini 2.5:

```
┌─────────────────┐
│  OddSpecAgent   │  Parse natural language ODD → structured spec
└────────┬────────┘
         │ odd_spec.json
         ▼
┌─────────────────────────────────────────┐
│  Parallel Sensor Analysis               │
│  ┌───────────┐ ┌──────────┐ ┌─────────┐ │
│  │Perception │ │  Motion  │ │Collision│ │
│  │  Agent    │ │  Agent   │ │  Agent  │ │
│  └───────────┘ └──────────┘ └─────────┘ │
└────────┬───────────┬───────────┬────────┘
         │ artifacts │           │
         ▼           ▼           ▼
┌─────────────────────────────────────────┐
│        EvaluatorAgent                   │  COD construction + compliance
└────────────────┬────────────────────────┘
                 │ cod_output.json
                 ▼
┌─────────────────────────────────────────┐
│          ReportAgent                    │  Executive summary
└─────────────────────────────────────────┘
```

Each agent:
- Calls **tool agents** that make VLM (Vision Language Model) calls
- Saves structured **artifacts** (JSON files)
- Performs **temporal analysis** across all windows

---

## 4. Define ODD Specification

Describe your robot's Operational Design Domain in natural language. The `OddSpecAgent` will parse this into a formal specification.

We'll use the default ODD for the Unitree Go2 robot:

In [ ]:
from odd_agents.odd_definition import DEFAULT_ODD_DESCRIPTION, ODD_DEFINITION_VERSION, ODD_SUMMARY

print(f"📋 ODD Definition v{ODD_DEFINITION_VERSION}")
print("="*60)
print(ODD_SUMMARY)
print("="*60)
print("\n💡 Full ODD description available in odd_description variable")

# Use the default ODD (you can customize this!)
odd_description = DEFAULT_ODD_DESCRIPTION

In [ ]:
# View the full ODD description
print(odd_description)

---

## 5. Select Scenario

Choose a preprocessed dataset to analyze. Scenarios contain:
- Camera images (PNG)
- LiDAR BEV images (occupancy, height, roughness)
- Motion/IMU data (JSON)
- Metadata

In [ ]:
# List available scenarios
data_dir = project_root / "data"

print("📁 Available Test Scenarios (2 windows - quick testing):")
test_dir = data_dir / "test"
if test_dir.exists():
    for scenario in sorted(test_dir.iterdir()):
        if scenario.is_dir() and not scenario.name.startswith('.'):
            windows = list(scenario.glob("w*"))
            print(f"   • {scenario.name} ({len(windows)} windows)")

print("\n📁 Available Production Scenarios (10+ windows):")
prod_dir = data_dir / "production" / "chunks"
if prod_dir.exists():
    for scenario in sorted(prod_dir.iterdir())[:10]:  # Show first 10
        if scenario.is_dir() and not scenario.name.startswith('.'):
            windows = list(scenario.glob("w*"))
            print(f"   • {scenario.name} ({len(windows)} windows)")
    remaining = len(list(prod_dir.iterdir())) - 10
    if remaining > 0:
        print(f"   ... and {remaining} more")

In [ ]:
# Select scenario to analyze
# Options: Use test scenarios for quick runs, production for full analysis

# Quick test (2 windows, ~$0.02, ~1 minute)
SCENARIO_NAME = "sim_2win"
SCENARIO_PATH = data_dir / "test" / SCENARIO_NAME

# Or use a production scenario (10+ windows, ~$0.10, ~5 minutes)
# SCENARIO_NAME = "sim_1_0_chunk_000_009"
# SCENARIO_PATH = data_dir / "production" / "chunks" / SCENARIO_NAME

# Or use real robot data
# SCENARIO_NAME = "real_173442_chunk_000_015" 
# SCENARIO_PATH = data_dir / "production" / "chunks" / SCENARIO_NAME

print(f"📂 Selected: {SCENARIO_NAME}")
print(f"   Path: {SCENARIO_PATH}")

if SCENARIO_PATH.exists():
    windows = sorted([w.name for w in SCENARIO_PATH.glob("w*") if w.is_dir()])
    print(f"   Windows: {len(windows)} ({windows[0]} to {windows[-1]})")
else:
    print("   ❌ Scenario not found! Check the path.")

In [ ]:
# Preview scenario data
from IPython.display import Image, display
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Show first window's data
first_window = SCENARIO_PATH / windows[0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Camera
cam_path = first_window / "camera.png"
if cam_path.exists():
    axes[0].imshow(mpimg.imread(str(cam_path)))
    axes[0].set_title("Camera")
    axes[0].axis('off')

# BEV channels
for i, channel in enumerate(["bev_occupancy.png", "bev_height.png", "bev_roughness.png"]):
    bev_path = first_window / channel
    if bev_path.exists():
        axes[i+1].imshow(mpimg.imread(str(bev_path)), cmap='gray')
        axes[i+1].set_title(channel.replace(".png", "").replace("bev_", "BEV ").title())
        axes[i+1].axis('off')

plt.suptitle(f"Window {windows[0]} - Sensor Data Preview", fontsize=14)
plt.tight_layout()
plt.show()

# Show motion data summary
motion_path = first_window / "motion.json"
if motion_path.exists():
    with open(motion_path) as f:
        motion = json.load(f)
    print(f"\n📊 Motion Data ({windows[0]}):")
    print(f"   Samples: {len(motion.get('timestamps', []))}")
    if motion.get('derived_speed'):
        speeds = motion['derived_speed']
        print(f"   Speed: {min(speeds):.3f} - {max(speeds):.3f} m/s")
    if motion.get('accel_x'):
        print(f"   IMU data: Available")
    else:
        print(f"   IMU data: Not available (using position-derived metrics)")

---

## 6. Run the Analysis Pipeline

Now we'll run the full 6-agent pipeline. This will:
1. Parse the ODD specification
2. Analyze all windows with Perception, Motion, and Collision agents
3. Construct the COD and check compliance
4. Generate an executive report

**Cost estimate:**
- 2 windows: ~$0.02
- 10 windows: ~$0.08-0.12
- 16 windows: ~$0.15-0.20

In [ ]:
# Configure models
# gemini-2.5-flash: Good balance of cost/quality (recommended)
# gemini-2.5-pro: Higher quality, 10x cost (use for Evaluator)

SENSOR_MODEL = "gemini-2.5-flash"  # For Perception, Motion, Collision
EVALUATOR_MODEL = "gemini-2.5-pro"  # For COD construction + compliance

print(f"🤖 Model Configuration:")
print(f"   Sensor agents: {SENSOR_MODEL}")
print(f"   Evaluator: {EVALUATOR_MODEL}")

In [ ]:
# Run the analysis pipeline
import asyncio
from datetime import datetime
from google import genai

from odd_agents.workflow import run_analysis_pipeline

print(f"🚀 Starting ODD Analysis Pipeline")
print(f"   Scenario: {SCENARIO_NAME}")
print(f"   Windows: {len(windows)}")
print(f"   Started: {datetime.now().strftime('%H:%M:%S')}")
print("="*60)

# Create Gemini client
genai_client = genai.Client(api_key=GOOGLE_API_KEY)

# Run the pipeline
result = await run_analysis_pipeline(
    scenario_path=str(SCENARIO_PATH),
    odd_description=odd_description,
    api_key=GOOGLE_API_KEY,
    genai_client=genai_client,
    sensor_model=SENSOR_MODEL,
    evaluator_model=EVALUATOR_MODEL,
    enable_knowledge=True,  # Seed knowledge docs for agents
)

print("="*60)
print(f"✅ Analysis complete! ({datetime.now().strftime('%H:%M:%S')})")

---

## 7. Explore Results

Let's examine what the agents discovered.

In [ ]:
# Compliance Verdict
compliance = result.get('compliance', {})

verdict = compliance.get('verdict', 'UNKNOWN')
confidence = compliance.get('confidence', 0)
stability = compliance.get('temporal_stability', 'UNKNOWN')

# Emoji based on verdict
verdict_emoji = {"IN_ODD": "✅", "BOUNDARY": "⚠️", "OUT_ODD": "❌"}.get(verdict, "❓")

print(f"{'='*60}")
print(f"📊 COMPLIANCE VERDICT")
print(f"{'='*60}")
print(f"")
print(f"   {verdict_emoji} {verdict}")
print(f"   Confidence: {confidence*100:.0f}%")
print(f"   Temporal Stability: {stability}")
print(f"")

# Critical axes
critical = compliance.get('critical_axes', [])
if critical:
    print(f"⚠️ Critical Axes:")
    for axis in critical:
        print(f"   • {axis}")

# Violations
violations = compliance.get('violations', [])
if violations:
    print(f"\n❌ Violations:")
    for v in violations[:5]:  # Show first 5
        print(f"   • {v}")

In [ ]:
# Per-Agent Summaries
print(f"{'='*60}")
print(f"🔍 AGENT SUMMARIES")
print(f"{'='*60}")

# Perception
perception = result.get('perception_summary', {})
if perception:
    env_type = perception.get('summary', {}).get('dominant_environment', 'unknown')
    data_source = perception.get('summary', {}).get('data_source', 'unknown')
    print(f"\n👁️ Perception:")
    print(f"   Environment: {env_type}")
    print(f"   Data source: {data_source}")

# Motion  
motion = result.get('motion_summary', {})
if motion:
    summary = motion.get('summary', {})
    max_speed = summary.get('max_speed_mps', 0)
    max_accel = summary.get('max_accel_mps2', 0)
    print(f"\n🏃 Motion:")
    print(f"   Max speed: {max_speed:.2f} m/s")
    print(f"   Max acceleration: {max_accel:.2f} m/s²")
    states = motion.get('temporal_analysis', {}).get('motion_state_sequence', [])
    if states:
        print(f"   States: {' → '.join(states[:5])}{'...' if len(states) > 5 else ''}")

# Collision
collision = result.get('collision_summary', {})
if collision:
    summary = collision.get('summary', {})
    collisions = summary.get('total_collisions_detected', 0)
    min_prox = summary.get('min_proximity_m', 0)
    print(f"\n💥 Collision:")
    print(f"   Collisions detected: {collisions}")
    print(f"   Minimum proximity: {min_prox:.2f}m")
    print(f"   (Advisory only - does not affect verdict)")

In [ ]:
# COD Region Summary
cod = result.get('cod', {})
cod_region = cod.get('cod_region', {})

if cod_region:
    print(f"{'='*60}")
    print(f"📐 COD REGION (Observed Conditions)")
    print(f"{'='*60}")
    
    for axis, data in cod_region.items():
        if isinstance(data, dict):
            status = data.get('status', 'UNKNOWN')
            status_emoji = {"IN_ODD": "✅", "BOUNDARY": "⚠️", "OUT_ODD": "❌"}.get(status, "")
            
            if 'observed_min' in data:
                print(f"\n{axis}: {status_emoji} {status}")
                print(f"   Observed: [{data.get('observed_min', 0):.2f}, {data.get('observed_max', 0):.2f}]")
                print(f"   ODD limit: [{data.get('odd_min', 0):.2f}, {data.get('odd_max', 0):.2f}]")
            elif 'observed_values' in data:
                print(f"\n{axis}: {status_emoji} {status}")
                print(f"   Observed: {data.get('observed_values', [])}")
                print(f"   Allowed: {data.get('odd_allowed', [])}")

In [ ]:
# Executive Report
report = result.get('report', {})

if report:
    print(f"{'='*60}")
    print(f"📝 EXECUTIVE REPORT")
    print(f"{'='*60}")
    
    exec_summary = report.get('executive_summary', '')
    if exec_summary:
        print(f"\n{exec_summary}")
    
    recommendations = report.get('recommendations', [])
    if recommendations:
        print(f"\n📋 Recommendations:")
        for i, rec in enumerate(recommendations, 1):
            print(f"   {i}. {rec}")

---

## 8. Visualize Results

In [ ]:
# Plot speed over windows
import matplotlib.pyplot as plt

# Get per-window motion data from artifacts
motion_artifact = result.get('artifacts', {}).get('motion_output.json', {})
per_window = motion_artifact.get('per_window', [])

if per_window:
    windows_ids = [w.get('window_id', f'w{i}') for i, w in enumerate(per_window)]
    max_speeds = [w.get('max_speed_mps', 0) for w in per_window]
    avg_speeds = [w.get('avg_speed_mps', 0) for w in per_window]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    
    x = range(len(windows_ids))
    ax.plot(x, max_speeds, 'b-o', label='Max Speed', linewidth=2, markersize=6)
    ax.plot(x, avg_speeds, 'g--s', label='Avg Speed', linewidth=1.5, markersize=4)
    
    # Add ODD limit line
    ax.axhline(y=2.5, color='r', linestyle='--', label='ODD Limit (2.5 m/s)', alpha=0.7)
    
    ax.set_xlabel('Window')
    ax.set_ylabel('Speed (m/s)')
    ax.set_title('Speed Profile Across Windows')
    ax.set_xticks(x)
    ax.set_xticklabels([w.replace('w0', 'w') for w in windows_ids], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No per-window motion data available")

In [ ]:
# Plot proximity over windows
collision_artifact = result.get('artifacts', {}).get('collision_output.json', {})
collision_per_window = collision_artifact.get('per_window', [])

if collision_per_window:
    windows_ids = [w.get('window_id', f'w{i}') for i, w in enumerate(collision_per_window)]
    proximities = [w.get('proximity_estimate_m', 2.5) for w in collision_per_window]
    collisions = [w.get('collision_detected', False) for w in collision_per_window]
    
    fig, ax = plt.subplots(figsize=(12, 4))
    
    x = range(len(windows_ids))
    colors = ['red' if c else 'green' for c in collisions]
    
    ax.bar(x, proximities, color=colors, alpha=0.7, edgecolor='black')
    
    # Add threshold lines
    ax.axhline(y=0.5, color='orange', linestyle='--', label='Caution (0.5m)', alpha=0.7)
    ax.axhline(y=0.3, color='red', linestyle='--', label='Danger (0.3m)', alpha=0.7)
    
    ax.set_xlabel('Window')
    ax.set_ylabel('Proximity (m)')
    ax.set_title('Obstacle Proximity per Window (🔴 = collision detected)')
    ax.set_xticks(x)
    ax.set_xticklabels([w.replace('w0', 'w') for w in windows_ids], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("No per-window collision data available")

---

## 9. Generate HTML Report

Create an interactive HTML report for sharing and documentation.

In [ ]:
# Save full results to JSON first
output_dir = project_root / "data" / "report_candidates" / datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir.mkdir(parents=True, exist_ok=True)

result_path = output_dir / SCENARIO_NAME / "full_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)

with open(result_path, 'w') as f:
    json.dump(result, f, indent=2, default=str)

print(f"💾 Results saved to: {result_path}")

In [ ]:
# Generate HTML report
import subprocess

html_output = project_root / "docs" / "reports" / f"{SCENARIO_NAME}_report.html"

cmd = [
    sys.executable,
    str(project_root / "scripts" / "generate_html_report.py"),
    "--input", str(result_path),
    "--scenario-dir", str(SCENARIO_PATH),
    "--output", str(html_output)
]

print(f"🔧 Generating HTML report...")
result_proc = subprocess.run(cmd, capture_output=True, text=True)

if result_proc.returncode == 0:
    print(f"✅ HTML report generated!")
    print(f"   📄 {html_output}")
    print(f"")
    print(f"💡 Open in browser or use: python -m http.server 8000")
else:
    print(f"❌ Error generating report:")
    print(result_proc.stderr)

---

## 10. Next Steps

### 🎯 Try Different Scenarios

1. Change `SCENARIO_NAME` in cell 5 to analyze different datasets
2. Compare simulation vs real robot data
3. Test edge cases with high-speed or cluttered scenarios

### 🔧 Customize the ODD

1. Edit `odd_description` in cell 4 to change constraints
2. Try stricter/looser limits and see how compliance changes
3. Add new axes (e.g., humidity, temperature) for custom domains

### 📊 Advanced Analysis

- Access raw artifacts: `result['artifacts']`
- Per-window data: `result['artifacts']['motion_output.json']['per_window']`
- Build custom visualizations from the data

### 📚 Documentation

- **Architecture**: `docs/architecture.html`
- **Agent docs**: `docs/agents/`
- **Knowledge docs**: `docs/agent_knowledge/`
- **Model guide**: `docs/MODEL_SELECTION_GUIDE.md`

---

**Questions?** Check the project documentation or explore the source code in `odd_agents/`!